In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Ensure numeric types for value columns
for df, col in [(df_metric, 'value'), (df_trace, 'value'), (df_log, 'value')]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Per rule 10: parse timestamp columns to UTC datetimes for internal use (but keep integer timestamps for output)
for df in (df_metric, df_trace, df_log, df_error):
    if 'timestamp' in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregation
if not df_metric.empty:
    metric_agg = df_metric.groupby(['cmdb_id', 'kpi_name']).agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        mean_value=('value', 'mean'),
        p95_value=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan)
    ).reset_index()
    metric_agg[['min_timestamp','max_timestamp']] = metric_agg[['min_timestamp','max_timestamp']].astype('Int64')
    metric_agg = metric_agg.sort_values('count', ascending=False).head(20)
    metric_ts_range = pd.DataFrame({
        'min_timestamp': [int(df_metric['timestamp'].min())],
        'max_timestamp': [int(df_metric['timestamp'].max())]
    })
else:
    metric_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','count','min_timestamp','max_timestamp','min_value','max_value','mean_value','p95_value'])
    metric_ts_range = pd.DataFrame({'min_timestamp': [], 'max_timestamp': []})

# 2) trace.csv aggregation
if not df_trace.empty:
    trace_agg = df_trace.groupby(['cmdb_id', 'trace_name']).agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        mean_value=('value', 'mean'),
        p95_value=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan)
    ).reset_index()
    trace_agg[['min_timestamp','max_timestamp']] = trace_agg[['min_timestamp','max_timestamp']].astype('Int64')
    trace_agg = trace_agg.sort_values('count', ascending=False).head(20)
    trace_ts_range = pd.DataFrame({
        'min_timestamp': [int(df_trace['timestamp'].min())],
        'max_timestamp': [int(df_trace['timestamp'].max())]
    })
else:
    trace_agg = pd.DataFrame(columns=['cmdb_id','trace_name','count','min_timestamp','max_timestamp','min_value','max_value','mean_value','p95_value'])
    trace_ts_range = pd.DataFrame({'min_timestamp': [], 'max_timestamp': []})

# 3) log.csv aggregation
if not df_log.empty:
    log_agg = df_log.groupby(['cmdb_id', 'log_name']).agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        mean_value=('value', 'mean'),
        p95_value=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan)
    ).reset_index()
    log_agg[['min_timestamp','max_timestamp']] = log_agg[['min_timestamp','max_timestamp']].astype('Int64')
    log_agg = log_agg.sort_values('count', ascending=False).head(20)
    log_ts_range = pd.DataFrame({
        'min_timestamp': [int(df_log['timestamp'].min())],
        'max_timestamp': [int(df_log['timestamp'].max())]
    })
else:
    log_agg = pd.DataFrame(columns=['cmdb_id','log_name','count','min_timestamp','max_timestamp','min_value','max_value','mean_value','p95_value'])
    log_ts_range = pd.DataFrame({'min_timestamp': [], 'max_timestamp': []})

# 4) error_logs.csv summary
error_total = len(df_error)
if error_total > 0:
    error_counts_by_cmdb = df_error.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).head(20)
    error_ts_range = pd.DataFrame({
        'min_timestamp': [int(df_error['timestamp'].min())],
        'max_timestamp': [int(df_error['timestamp'].max())]
    })
    error_samples = df_error.sort_values('timestamp', ascending=False).head(5)[['timestamp', 'cmdb_id', 'message']]
else:
    error_counts_by_cmdb = pd.DataFrame(columns=['cmdb_id','count'])
    error_ts_range = pd.DataFrame({'min_timestamp': [], 'max_timestamp': []})
    error_samples = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Display compact results (tables with headers). Limit rows already enforced.
metric_agg, metric_ts_range, trace_agg, trace_ts_range, log_agg, log_ts_range, pd.DataFrame({'error_total':[error_total]}), error_counts_by_cmdb, error_ts_range, error_samples

```
Out[1]:
```
summary = (
    "Summary of telemetry (UTC):\n"
    "- metric.csv: data present from 1705601580 to 1705603020. Top metric series by sample count are for 'carts' and related DBs. Notable values:\n"
    "  * carts: cpu mean ~0.91 (P95 ~1.43), mem ~2.14e8, workload max ~8.42 (P95 ~8.29), socket P95 ~12.28.\n"
    "  * carts-db: diskio shows very large values (max ~2.3e6, P95 ~2.23e6) and cpu P95 ~6.42.\n"
    "  * catalogue-db: mem ~4.32e8 (P95 ~4.324e8).\n"
    "  * front-end has an 'error' metric with P95 ~0.805.\n"
    "- trace.csv: no trace data available.\n"
    "- log.csv: data present from 1705601580 to 1705603020. Key log series and observations:\n"
    "  * user and front-end have very large log.row_count values (user row_count max 619, P95 ~612; front-end max 2105, P95 ~2028).\n"
    "  * Many services show log.error_count series but counts are zero in these samples.\n"
    "  * Several services (payment, orders, shipping, queue-master, etc.) also show row_count activity with varying magnitudes.\n"
    "- error_logs.csv: no rows (total 0). No raw error message samples are available.\n\n"
    "Bottom line: metrics highlight heavy resource or activity signals for 'carts' and 'carts-db' (high memory/diskio/workload/socket/cpu), and significant request/log volume in 'front-end' and 'user'. No trace data and no recorded error log entries to correlate with these signals."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id  kpi_name  count  min_timestamp  max_timestamp     min_value     max_value    mean_value     p95_value
0          carts       cpu     25     1705601580     1705603020  1.348526e-01  1.436683e+00  9.141259e-01  1.427343e+00
3          carts       mem     25     1705601580     1705603020  2.141048e+08  2.141950e+08  2.141321e+08  2.141597e+08
5          carts  workload     25     1705601580     1705603020  0.000000e+00  8.424533e+00  5.187625e+00  8.293107e+00
4          carts    socket     25     1705601580     1705603020  8.000000e+00  1.290000e+01  1.085683e+01  1.228000e+01
8       carts-db       mem     25     1705601580     1705603020  1.001935e+08  1.021062e+08  1.012765e+08  1.020983e+08
9       carts-db    socket     25     1705601580     1705603020  6.000000e+00  7.000000e+00  6.601333e+00  7.000000e+00
6       carts-db       cpu     25     1705601580     1705603020  3.837776e-01  6.761426e+00  4.133686e+00  6.420331e+00
7       carts-db    diskio     25     1705601580     1705603020  2.945209e+05  2.299440e+06  1.608540e+06  2.229646e+06
10     catalogue       cpu     25     1705601580     1705603020  1.338964e-03  1.925225e-01  1.181992e-01  1.840003e-01
17  catalogue-db       cpu     25     1705601580     1705603020  6.403300e-02  2.156455e-01  1.569108e-01  2.094876e-01
14     catalogue       mem     25     1705601580     1705603020  5.543868e+06  1.102070e+07  7.159080e+06  9.033482e+06
15     catalogue    socket     25     1705601580     1705603020  6.000000e+00  1.876667e+01  8.956667e+00  1.457333e+01
60      shipping       cpu     25     1705601580     1705603020  1.045987e-01  5.872516e-01  2.281520e-01  2.926204e-01
63      shipping       mem     25     1705601580     1705603020  3.060263e+08  3.063977e+08  3.063595e+08  3.063830e+08
65      shipping  workload     25     1705601580     1705603020  0.000000e+00  2.121017e+00  1.262751e+00  2.100790e+00
16     catalogue  workload     25     1705601580     1705603020  0.000000e+00  4.222550e+00  2.667431e+00  4.193797e+00
18  catalogue-db    diskio     25     1705601580     1705603020  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
19  catalogue-db       mem     25     1705601580     1705603020  4.309565e+08  4.324434e+08  4.314935e+08  4.324423e+08
22     front-end     error     25     1705601580     1705603020  0.000000e+00  8.700333e-01  1.293400e-01  8.053833e-01
20  catalogue-db    socket     25     1705601580     1705603020  2.000000e+00  4.316667e+00  3.620000e+00  4.000000e+00,    min_timestamp  max_timestamp
0     1705601580     1705603020, Empty DataFrame
Columns: [cmdb_id, trace_name, count, min_timestamp, max_timestamp, min_value, max_value, mean_value, p95_value]
Index: [], Empty DataFrame
Columns: [min_timestamp, max_timestamp]
Index: [],          cmdb_id         log_name  count  min_timestamp  max_timestamp  min_value  max_value   mean_value  p95_value
20          user  log.error_count     25     1705601580     1705603020          0          0     0.000000       0.00
21          user    log.row_count     25     1705601580     1705603020          8        619   373.880000     612.40
13       payment    log.row_count     25     1705601580     1705603020          8        107    75.200000     104.00
12       payment  log.error_count     25     1705601580     1705603020          0          0     0.000000       0.00
8      front-end  log.error_count     25     1705601580     1705603020          0          0     0.000000       0.00
9      front-end    log.row_count     25     1705601580     1705603020         11       2105  1250.960000    2028.20
2       carts-db  log.error_count     24     1705601580     1705602960          0          0     0.000000       0.00
3       carts-db    log.row_count     24     1705601580     1705602960          1          2     1.041667       1.00
5      catalogue    log.row_count     21     1705601580     1705602840          2        168   125.095238     167.00
4      catalogue  log.error_count     21     1705601580     1705602840          0          0     0.000000       0.00
19      shipping    log.row_count     18     1705601580     1705602720          6         67    51.111111      64.45
18      shipping  log.error_count     18     1705601580     1705602720          0          0     0.000000       0.00
11        orders    log.row_count     18     1705601580     1705602720         12        134   102.222222     128.90
10        orders  log.error_count     18     1705601580     1705602720          0          0     0.000000       0.00
14  queue-master  log.error_count     18     1705601580     1705602720          0          0     0.000000       0.00
15  queue-master    log.row_count     18     1705601580     1705602720         54        603   460.000000     580.05
0          carts  log.error_count     16     1705601580     1705602600          0          0     0.000000       0.00
1          carts    log.row_count     16     1705601580     1705602600         25        150    68.750000     131.25
17    session-db    log.row_count      5     1705601640     1705602840          5          5     5.000000       5.00
16    session-db  log.error_count      5     1705601640     1705602840          0          0     0.000000       0.00,    min_timestamp  max_timestamp
0     1705601580     1705603020,    error_total
0            0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [min_timestamp, max_timestamp]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if available, otherwise load
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")

# Normalize log dataframe to have kpi_name column to unify processing
df_log2 = df_log.copy()
if 'log_name' in df_log2.columns:
    df_log2 = df_log2.rename(columns={'log_name': 'kpi_name'})
else:
    df_log2['kpi_name'] = df_log2.get('kpi_name', None)

# Metric dataframe already has kpi_name
df_metric2 = df_metric.copy()

# Combine datasets and add a source column
df_metric2['__source'] = 'metric'
df_log2['__source'] = 'log'
# Keep only required columns (timestamp, cmdb_id, kpi_name, value, __source)
df_all = pd.concat([
    df_metric2[['timestamp', 'cmdb_id', 'kpi_name', 'value', '__source']],
    df_log2[['timestamp', 'cmdb_id', 'kpi_name', 'value', '__source']]
], ignore_index=True)

# Ensure numeric values
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')

# Parse timestamps to UTC datetimes for filtering
df_all['timestamp_dt'] = pd.to_datetime(df_all['timestamp'], unit='s', utc=True)

# Compute global P95 and P05 per (cmdb_id, kpi_name) using full series BEFORE filtering (rule 9)
group_cols = ['cmdb_id', 'kpi_name']
g = df_all.groupby(group_cols)['value']
group_stats = g.agg(global_p95=lambda x: x.quantile(0.95),
                    global_p05=lambda x: x.quantile(0.05),
                    total_count='count').reset_index()

# Merge thresholds back to per-sample rows
df_all = df_all.merge(group_stats, on=group_cols, how='left')

# Incident window (UTC) - parse as UTC-aware timestamps
window_start = pd.to_datetime('2024-01-18 18:10:12', utc=True)
window_end = pd.to_datetime('2024-01-18 18:40:12', utc=True)

# Filter to window
df_window = df_all[(df_all['timestamp_dt'] >= window_start) & (df_all['timestamp_dt'] <= window_end)].copy()

# Determine which KPI names are eligible for low-anomaly detection
low_kpi_tokens = ['workload','row_count','throughput','request','requests','rps']

# Lowercase kpi_name for matching
df_window['kpi_name_lower'] = df_window['kpi_name'].astype(str).str.lower()

# Flag high and low anomalies per sample (using global thresholds)
df_window['high_flag'] = df_window['value'] >= df_window['global_p95']
df_window['low_flag'] = df_window.apply(
    lambda row: (any(tok in row['kpi_name_lower'] for tok in low_kpi_tokens) and pd.notna(row['value']) 
                 and pd.notna(row['global_p05']) and row['value'] <= row['global_p05']),
    axis=1
)
df_window['anomaly_flag'] = df_window['high_flag'] | df_window['low_flag']

# Aggregate per series within window
agg_window = df_window.groupby(group_cols).agg(
    window_count=('value', 'count'),
    anomaly_count=('anomaly_flag', 'sum'),
    first_anomaly_timestamp=('timestamp', lambda x: int(x[df_window.loc[x.index, 'anomaly_flag']].min()) if df_window.loc[x.index, 'anomaly_flag'].any() else np.nan),
    min_value_in_window=('value', 'min'),
    max_value_in_window=('value', 'max'),
    mean_value_in_window=('value', 'mean'),
    min_timestamp_in_window=('timestamp', 'min'),
    max_timestamp_in_window=('timestamp', 'max')
).reset_index()

# Merge global thresholds into aggregated window results
agg_window = agg_window.merge(group_stats, on=group_cols, how='left')

# Determine anomaly_type and severity per series
def determine_type_and_severity(row):
    has_high = False
    has_low = False
    # We need to inspect sample-level flags for this series to know if high/low occurred.
    # Extract the subset quickly
    subset = df_window[(df_window['cmdb_id'] == row['cmdb_id']) & (df_window['kpi_name'] == row['kpi_name'])]
    if subset.empty or row['anomaly_count'] == 0:
        return pd.Series({'anomaly_type': None, 'severity': np.nan})
    has_high = subset['high_flag'].any()
    has_low = subset['low_flag'].any()
    # Compute severity(s) safely
    sev_high = np.nan
    sev_low = np.nan
    if has_high and pd.notna(row['global_p95']) and row['global_p95'] != 0:
        sev_high = row['max_value_in_window'] / row['global_p95']
    if has_low and pd.notna(row['global_p05']) and row['min_value_in_window'] != 0:
        # Note: if min_value_in_window is 0, set severity to inf to indicate extreme drop
        if row['min_value_in_window'] == 0:
            sev_low = np.inf
        else:
            sev_low = row['global_p05'] / row['min_value_in_window']
    if has_high and has_low:
        # both -> choose max severity and label 'both'
        sevs = [s for s in (sev_high, sev_low) if pd.notna(s)]
        severity = np.nanmax(sevs) if sevs else np.nan
        return pd.Series({'anomaly_type': 'both', 'severity': severity})
    if has_high:
        return pd.Series({'anomaly_type': 'high', 'severity': sev_high})
    if has_low:
        return pd.Series({'anomaly_type': 'low', 'severity': sev_low})
    return pd.Series({'anomaly_type': None, 'severity': np.nan})

type_sev = agg_window.apply(determine_type_and_severity, axis=1)
agg_window = pd.concat([agg_window, type_sev], axis=1)

# Filter to series that have at least one anomaly
anomalies = agg_window[agg_window['anomaly_count'] > 0].copy()

# Prepare final columns and convert first_anomaly_timestamp to integer where present
final_cols = [
    'cmdb_id', 'kpi_name', 'global_p95', 'global_p05',
    'window_count', 'anomaly_count', 'first_anomaly_timestamp',
    'min_value_in_window', 'max_value_in_window', 'mean_value_in_window',
    'severity', 'anomaly_type'
]
# Ensure first_anomaly_timestamp is int dtype if not null
anomalies['first_anomaly_timestamp'] = anomalies['first_anomaly_timestamp'].apply(lambda x: int(x) if pd.notna(x) else x)

# Sort by severity desc and limit to top 20
anomalies_sorted = anomalies.sort_values('severity', ascending=False).reset_index(drop=True)
anomalies_top20 = anomalies_sorted[final_cols].head(20)

# Metrics: total distinct series scanned and number with anomalies
# Distinct series scanned across both metric and log (using group_stats)
total_series_scanned = int(group_stats.shape[0])
series_with_anomalies = int(anomalies.shape[0])

# Compact outputs
anomalies_top20, pd.DataFrame({'total_series_scanned':[total_series_scanned], 'series_with_anomalies':[series_with_anomalies]})

```
Out[2]:
```
summary = (
    "Incident-window summary (2024-01-18 18:10:12 UTC to 2024-01-18 18:40:12 UTC):\n"
    "- Scanned series: 98 distinct (cmdb_id, kpi_name). Series with anomalies in window: 98.\n"
    "- Top anomalous series by severity (severity = ratio as computed):\n"
    "  1) catalogue / log.row_count — severity 14.0 (type: both: spikes above P95 and dips ≤ P05)\n"
    "  2) payment / log.row_count — severity 5.0 (both)\n"
    "  3) user / log.row_count — severity 5.0 (both)\n"
    "  4) catalogue / error — severity ~2.23 (high spike)\n"
    "  5) shipping / cpu — severity ~2.01 (high spike)\n"
    "- Other notable affected series: carts-db log.row_count (severity 2.0, all samples anomalous), front-end log.row_count (severity ~1.82, both), orders and queue-master row_count (severity ~1.85, both), several catalogue KPIs (latency, socket, mem) show high anomalies.\n"
    "- Interpretation: many request/row_count style KPIs (front-end, user, payment, catalogue, carts, queue-master, etc.) show both high and low anomalies in the window indicating volatile traffic/load (spikes above historical P95 and dips below P05). Several resource KPIs (cpu, mem, diskio, socket) also had high spikes (e.g., shipping CPU, catalogue mem, carts-db diskio).\n"
    "- Actionable note: investigate traffic patterns and load balancing for front-end/user/payment/catalogue/carts services and check carts-db and catalogue-db resource pressure (disk I/O and memory). Correlate with application traces or logs (no trace correlation provided here) to pinpoint causes."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id       kpi_name    global_p95    global_p05  window_count  anomaly_count  first_anomaly_timestamp  min_value_in_window  max_value_in_window  mean_value_in_window   severity anomaly_type
0           catalogue  log.row_count  1.670000e+02  2.800000e+01            21              5               1705601880         2.000000e+00         1.680000e+02          1.250952e+02  14.000000         both
1             payment  log.row_count  1.040000e+02  4.000000e+01            25             10               1705601940         8.000000e+00         1.070000e+02          7.520000e+01   5.000000         both
2                user  log.row_count  6.124000e+02  4.000000e+01            25              7               1705601940         8.000000e+00         6.190000e+02          3.738800e+02   5.000000         both
3           catalogue          error  2.748733e-01  0.000000e+00            13              1               1705602840         0.000000e+00         6.121833e-01          5.384231e-02   2.227147         high
4            shipping            cpu  2.926204e-01  1.073018e-01            25              2               1705601640         1.045987e-01         5.872516e-01          2.281520e-01   2.006872         high
5            carts-db  log.row_count  1.000000e+00  1.000000e+00            24             24               1705601580         1.000000e+00         2.000000e+00          1.041667e+00   2.000000         both
6              orders  log.row_count  1.289000e+02  2.220000e+01            18              2               1705601940         1.200000e+01         1.340000e+02          1.022222e+02   1.850000         both
7            shipping  log.row_count  6.445000e+01  1.110000e+01            18              2               1705601940         6.000000e+00         6.700000e+01          5.111111e+01   1.850000         both
8        queue-master  log.row_count  5.800500e+02  9.990000e+01            18              2               1705601940         5.400000e+01         6.030000e+02          4.600000e+02   1.850000         both
9           front-end  log.row_count  2.028200e+03  2.000000e+01            25              4               1705601880         1.100000e+01         2.105000e+03          1.250960e+03   1.818182         both
10             orders            cpu  1.055974e+00  1.421970e-01            25              2               1705601640         1.404156e-01         1.741316e+00          6.913822e-01   1.649015         high
11       queue-master            cpu  3.139447e-01  1.184559e-01            25              2               1705601640         1.182577e-01         5.151079e-01          2.353402e-01   1.640760         high
12          catalogue     latency-90  1.686285e+00  4.619474e-03            22              2               1705602480         4.606663e-03         2.638337e+00          5.585773e-01   1.564585         high
13          catalogue     latency-50  2.592938e-01  3.010819e-03            22              2               1705602480         3.003702e-03         3.764205e-01          8.112652e-02   1.451714         high
14          catalogue         socket  1.457333e+01  6.000000e+00            25              2               1705602540         6.000000e+00         1.876667e+01          8.956667e+00   1.287740         high
15          catalogue            mem  9.033482e+06  5.635454e+06            25              2               1705602480         5.543868e+06         1.102070e+07          7.159080e+06   1.219983         high
16       queue-master         diskio  4.075420e+04  1.211831e+04            25              2               1705601700         1.135418e+04         4.710345e+04          2.600708e+04   1.155794         high
17              carts  log.row_count  1.312500e+02  2.500000e+01            16              5               1705601580         2.500000e+01         1.500000e+02          6.875000e+01   1.142857         both
18  rabbitmq-exporter            cpu  1.272777e-04  6.591186e-05            25              2               1705601880         5.694433e-05         1.412541e-04          8.987407e-05   1.109810         high
19          front-end          error  8.053833e-01  0.000000e+00            25              2               1705602300         0.000000e+00         8.700333e-01          1.293400e-01   1.080272         high,    total_series_scanned  series_with_anomalies
0                    98                     98)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Ensure metric and log dataframes are loaded (reuse if available)
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")

# Normalize log to kpi_name
df_log2 = df_log.copy()
if 'log_name' in df_log2.columns:
    df_log2 = df_log2.rename(columns={'log_name': 'kpi_name'})
else:
    df_log2['kpi_name'] = df_log2.get('kpi_name', None)

# Combine metric and log series (we only need timestamp, cmdb_id, kpi_name, value)
df_metric2 = df_metric.copy()
df_metric2['kpi_name'] = df_metric2['kpi_name'].astype(str)
df_log2['kpi_name'] = df_log2['kpi_name'].astype(str)

df_all = pd.concat([
    df_metric2[['timestamp', 'cmdb_id', 'kpi_name', 'value']].assign(__source='metric'),
    df_log2[['timestamp', 'cmdb_id', 'kpi_name', 'value']].assign(__source='log')
], ignore_index=True)

# Ensure numeric values
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')
df_all['timestamp'] = pd.to_numeric(df_all['timestamp'], errors='coerce').astype('Int64')

# Compute global P95 and P05 per (cmdb_id, kpi_name) using full series (reuse if exists)
if 'group_stats_full' in globals():
    group_stats = group_stats_full
else:
    group_stats = df_all.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p05=lambda x: x.quantile(0.05),
        total_count='count'
    ).reset_index()
    group_stats_full = group_stats  # cache

# Incident window (UTC) - parse as UTC-aware datetimes
window_start = pd.to_datetime('2024-01-18 18:10:12', utc=True)
window_end = pd.to_datetime('2024-01-18 18:40:12', utc=True)

# Parse timestamps to datetime for filtering
df_all['timestamp_dt'] = pd.to_datetime(df_all['timestamp'].astype('int64'), unit='s', utc=True)

# Candidate components
candidates = ["catalogue","carts","orders","payment","user"]

# Filter to candidate components
df_cand = df_all[df_all['cmdb_id'].isin(candidates)].copy()

# Merge global thresholds into df_cand
df_cand = df_cand.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

# Define low-anomaly tokens
low_kpi_tokens = ['workload','row_count','throughput','request','requests','rps']

# Filter to incident window
df_win = df_cand[(df_cand['timestamp_dt'] >= window_start) & (df_cand['timestamp_dt'] <= window_end)].copy()
if df_win.empty:
    selected_rows = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_type','first_anomaly_timestamp','value_at_first_anomaly',
        'global_p95','global_p05','max_value_in_window','min_value_in_window','mean_value_in_window',
        'anomaly_count_in_window','longest_consecutive_anomaly_run','severity'
    ])
    summary_line = "No data in incident window for candidate components."
    selected_rows, summary_line
else:
    # Lowercase kpi_name for token matching
    df_win['kpi_lower'] = df_win['kpi_name'].str.lower()

    # Flags per sample
    df_win['high_flag'] = df_win['value'] >= df_win['global_p95']
    df_win['low_flag'] = df_win.apply(
        lambda r: (any(tok in r['kpi_lower'] for tok in low_kpi_tokens)
                   and pd.notna(r['value']) and pd.notna(r['global_p05'])
                   and r['value'] <= r['global_p05']),
        axis=1
    )
    df_win['anom_flag'] = df_win['high_flag'] | df_win['low_flag']

    # Aggregate per series within window
    agg = df_win.groupby(['cmdb_id','kpi_name']).agg(
        window_count=('value','count'),
        anomaly_count_in_window=('anom_flag','sum'),
        first_anom_ts=('timestamp', lambda x: int(x[df_win.loc[x.index,'anom_flag']].min()) if df_win.loc[x.index,'anom_flag'].any() else pd.NA),
        # value at first anomaly: need to pick value where timestamp equals first_anom_ts
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        mean_value_in_window=('value','mean'),
        min_timestamp_in_window=('timestamp','min'),
        max_timestamp_in_window=('timestamp','max')
    ).reset_index()

    # Get value_at_first_anomaly by joining back
    # Create mapping of (cmdb_id,kpi_name,first_anom_ts) -> value
    def get_value_at_first(row):
        if pd.isna(row['first_anom_ts']):
            return np.nan
        subset = df_win[
            (df_win['cmdb_id']==row['cmdb_id']) &
            (df_win['kpi_name']==row['kpi_name']) &
            (df_win['timestamp']==int(row['first_anom_ts']))
        ]
        if subset.empty:
            # fallback: take earliest anomaly row by timestamp
            subset2 = df_win[
                (df_win['cmdb_id']==row['cmdb_id']) &
                (df_win['kpi_name']==row['kpi_name']) &
                (df_win['anom_flag'])
            ].sort_values('timestamp')
            return float(subset2.iloc[0]['value']) if not subset2.empty else np.nan
        return float(subset.iloc[0]['value'])

    agg['value_at_first_anomaly'] = agg.apply(get_value_at_first, axis=1)

    # Merge global thresholds
    agg = agg.merge(group_stats[['cmdb_id','kpi_name','global_p95','global_p05']], on=['cmdb_id','kpi_name'], how='left')

    # Compute longest consecutive anomaly run per series
    def longest_run(row):
        subset = df_win[(df_win['cmdb_id']==row['cmdb_id']) & (df_win['kpi_name']==row['kpi_name'])].sort_values('timestamp')
        anom = subset[subset['anom_flag']].copy()
        if anom.empty:
            return 0
        # compute diffs in seconds
        anom['diff'] = anom['timestamp'].diff().fillna(9999).astype(int)
        # start new run when diff != 60
        anom['run_id'] = (anom['diff'] != 60).cumsum()
        run_lengths = anom.groupby('run_id').size()
        return int(run_lengths.max())

    agg['longest_consecutive_anomaly_run'] = agg.apply(longest_run, axis=1).astype(int)

    # Determine anomaly_type per series (high/low/both/none)
    def determine_type(row):
        subset = df_win[(df_win['cmdb_id']==row['cmdb_id']) & (df_win['kpi_name']==row['kpi_name'])]
        if subset.empty or subset['anom_flag'].sum()==0:
            return None
        has_high = subset['high_flag'].any()
        has_low = subset['low_flag'].any()
        if has_high and has_low:
            return 'both'
        if has_high:
            return 'high'
        if has_low:
            return 'low'
        return None

    agg['anomaly_type'] = agg.apply(determine_type, axis=1)

    # Compute severity for tie-breaker: for high = max_value_in_window/global_p95; for low = global_p05/min_value_in_window; for both take max of both (handle div by zero)
    def compute_sev(row):
        sev_high = np.nan
        sev_low = np.nan
        if row['anomaly_type'] in ('high','both'):
            if pd.notna(row['global_p95']) and row['global_p95'] != 0:
                sev_high = row['max_value_in_window'] / row['global_p95']
        if row['anomaly_type'] in ('low','both'):
            if pd.notna(row['global_p05']):
                if row['min_value_in_window'] == 0:
                    sev_low = np.inf
                else:
                    sev_low = row['global_p05'] / row['min_value_in_window']
        # choose max of numeric severities
        sevs = [s for s in (sev_high, sev_low) if pd.notna(s)]
        return float(np.nanmax(sevs)) if sevs else np.nan

    agg['severity'] = agg.apply(compute_sev, axis=1)

    # Keep only series that have at least one anomaly in window
    agg_anom = agg[agg['anomaly_count_in_window'] > 0].copy()

    # For each component, choose the series with earliest first_anomaly_timestamp; tie-breaker: higher severity
    # Remove rows with NA first_anom_ts just in case
    agg_anom = agg_anom[pd.notna(agg_anom['first_anom_ts'])].copy()
    # Ensure types
    agg_anom['first_anom_ts'] = agg_anom['first_anom_ts'].astype(int)

    # Sort by first_anom_ts asc, severity desc to apply grouping pick
    agg_anom_sorted = agg_anom.sort_values(['cmdb_id','first_anom_ts','severity'], ascending=[True, True, False]).copy()

    # For each cmdb_id pick the row with minimal first_anom_ts; if multiple with same ts pick max severity
    selected = agg_anom_sorted.groupby('cmdb_id', sort=False).apply(
        lambda df: df.loc[df['first_anom_ts'] == df['first_anom_ts'].min()].sort_values('severity', ascending=False).head(1)
    ).reset_index(drop=True)

    # Limit to requested candidate components order and up to 5 rows
    selected = selected[selected['cmdb_id'].isin(candidates)].copy()
    # Sort the selected rows by first_anom_ts ascending
    selected = selected.sort_values('first_anom_ts').head(5)

    # Prepare final output columns
    selected_rows = selected[[
        'cmdb_id','kpi_name','anomaly_type','first_anom_ts','value_at_first_anomaly',
        'global_p95','global_p05','max_value_in_window','min_value_in_window','mean_value_in_window',
        'anomaly_count_in_window','longest_consecutive_anomaly_run','severity'
    ]].rename(columns={'first_anom_ts':'first_anomaly_timestamp'})

    # Small summary: earliest overall anomaly timestamp among these components and which component produced it
    if not selected_rows.empty:
        earliest_idx = selected_rows['first_anomaly_timestamp'].idxmin()
        earliest_ts = int(selected_rows.loc[earliest_idx,'first_anomaly_timestamp'])
        earliest_comp = selected_rows.loc[earliest_idx,'cmdb_id']
        summary_line = f"Earliest anomaly among selected components: timestamp {earliest_ts} (UTC) produced by component '{earliest_comp}'."
    else:
        summary_line = "No anomalies detected for candidate components in the window."

    # Ensure compactness: display selected_rows with limited columns
    selected_rows, summary_line

```
Out[3]:
```
I could not produce the requested summary because the last code cell returned no result (None). That means the final variables with the selected rows and summary line were not emitted back from the executed cell.

Next steps I can take for you (choose one):
- Rerun the selection step and explicitly return the selected_rows and summary_line so you get the 5-row table and the earliest-anomaly summary.
- Or I can modify the code to print a compact CSV-like table and the summary line immediately.

Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse loaded dataframes if present, else load from CSV
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")

# Normalize log to kpi_name column
df_log2 = df_log.copy()
if 'log_name' in df_log2.columns:
    df_log2 = df_log2.rename(columns={'log_name': 'kpi_name'})
else:
    df_log2['kpi_name'] = df_log2.get('kpi_name', None)

# Combine metric and log series
df_metric2 = df_metric.copy()
df_metric2['kpi_name'] = df_metric2['kpi_name'].astype(str)
df_log2['kpi_name'] = df_log2['kpi_name'].astype(str)

df_all = pd.concat([
    df_metric2[['timestamp','cmdb_id','kpi_name','value']].assign(__source='metric'),
    df_log2[['timestamp','cmdb_id','kpi_name','value']].assign(__source='log')
], ignore_index=True)

# Ensure numeric types
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')
df_all['timestamp'] = pd.to_numeric(df_all['timestamp'], errors='coerce').astype('Int64')

# Parse timestamps to UTC datetimes for filtering (rule 10)
df_all['timestamp_dt'] = pd.to_datetime(df_all['timestamp'].astype('int64'), unit='s', utc=True)

# Candidate components
candidates = ["catalogue","carts","orders","payment","user"]

# Compute global thresholds (P95, P05) per (cmdb_id, kpi_name) using full series (rule 9)
group_stats = df_all.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_p95=lambda x: x.quantile(0.95),
    global_p05=lambda x: x.quantile(0.05)
).reset_index()

# Incident window (UTC)
window_start = pd.to_datetime('2024-01-18 18:10:12', utc=True)
window_end = pd.to_datetime('2024-01-18 18:40:12', utc=True)

# Filter to candidate components and incident window
df_cand = df_all[df_all['cmdb_id'].isin(candidates)].copy()
df_cand = df_cand.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

df_win = df_cand[(df_cand['timestamp_dt'] >= window_start) & (df_cand['timestamp_dt'] <= window_end)].copy()

# If no data in window, return empty results and message
if df_win.empty:
    selected_rows = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_type','first_anomaly_timestamp','value_at_first_anomaly',
        'global_p95','global_p05','max_value_in_window','min_value_in_window','mean_value_in_window',
        'anomaly_count_in_window','longest_consecutive_anomaly_run'
    ])
    summary_line = "No data in incident window for candidate components."
    selected_rows, summary_line
else:
    # Define low-anomaly KPI tokens
    low_kpi_tokens = ['workload','row_count','throughput','request','requests','rps']

    # Lowercase kpi_name for token matching
    df_win['kpi_lower'] = df_win['kpi_name'].str.lower()

    # Sample-level anomaly flags
    df_win['high_flag'] = df_win['value'] >= df_win['global_p95']
    df_win['low_flag'] = df_win.apply(
        lambda r: (any(tok in r['kpi_lower'] for tok in low_kpi_tokens)
                   and pd.notna(r['value']) and pd.notna(r['global_p05'])
                   and r['value'] <= r['global_p05']),
        axis=1
    )
    df_win['anom_flag'] = df_win['high_flag'] | df_win['low_flag']

    # Aggregate per series within window
    grouped = df_win.groupby(['cmdb_id','kpi_name'])

    agg = grouped.agg(
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        mean_value_in_window=('value','mean'),
        anomaly_count_in_window=('anom_flag','sum'),
        window_count=('value','count')
    ).reset_index()

    # First anomaly timestamp and value_at_first_anomaly
    def compute_first_anom_info(row):
        subset = df_win[(df_win['cmdb_id']==row['cmdb_id']) & (df_win['kpi_name']==row['kpi_name']) & (df_win['anom_flag'])]
        if subset.empty:
            return pd.Series({'first_anomaly_timestamp': pd.NA, 'value_at_first_anomaly': np.nan, 'anomaly_type': None})
        # earliest anomaly by timestamp
        first_row = subset.sort_values('timestamp').iloc[0]
        # determine anomaly_type for series
        has_high = subset['high_flag'].any()
        has_low = subset['low_flag'].any()
        if has_high and has_low:
            atype = 'both'
        elif has_high:
            atype = 'high'
        elif has_low:
            atype = 'low'
        else:
            atype = None
        return pd.Series({
            'first_anomaly_timestamp': int(first_row['timestamp']),
            'value_at_first_anomaly': float(first_row['value']),
            'anomaly_type': atype
        })

    first_info = agg.apply(compute_first_anom_info, axis=1)
    agg = pd.concat([agg, first_info], axis=1)

    # Merge global thresholds into agg
    agg = agg.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

    # Compute longest consecutive anomaly run per series
    def longest_consecutive_run(row):
        subset = df_win[(df_win['cmdb_id']==row['cmdb_id']) & (df_win['kpi_name']==row['kpi_name'])].sort_values('timestamp')
        if subset.empty:
            return 0
        # mark anomaly rows and compute runs where consecutive timestamps differ by 60s
        anom = subset[subset['anom_flag']].copy()
        if anom.empty:
            return 0
        anom['diff'] = anom['timestamp'].diff().fillna(9999).astype(int)
        anom['run_id'] = (anom['diff'] != 60).cumsum()
        run_lengths = anom.groupby('run_id').size()
        return int(run_lengths.max())

    agg['longest_consecutive_anomaly_run'] = agg.apply(longest_consecutive_run, axis=1).astype(int)

    # Keep only series with at least one anomaly
    agg_anom = agg[agg['anomaly_count_in_window'] > 0].copy()
    # If no anomalies, return accordingly
    if agg_anom.empty:
        selected_rows = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','anomaly_type','first_anomaly_timestamp','value_at_first_anomaly',
            'global_p95','global_p05','max_value_in_window','min_value_in_window','mean_value_in_window',
            'anomaly_count_in_window','longest_consecutive_anomaly_run'
        ])
        summary_line = "No anomalies detected for candidate components in the window."
        selected_rows, summary_line
    else:
        # Compute severity for tie-breaking
        def compute_severity(row):
            sev_high = np.nan
            sev_low = np.nan
            if row['anomaly_type'] in ('high','both'):
                if pd.notna(row['global_p95']) and row['global_p95'] != 0:
                    sev_high = row['max_value_in_window'] / row['global_p95']
            if row['anomaly_type'] in ('low','both'):
                if pd.notna(row['global_p05']):
                    if row['min_value_in_window'] == 0:
                        sev_low = np.inf
                    else:
                        sev_low = row['global_p05'] / row['min_value_in_window']
            sevs = [s for s in (sev_high, sev_low) if pd.notna(s)]
            return float(np.nanmax(sevs)) if sevs else np.nan

        agg_anom['severity'] = agg_anom.apply(compute_severity, axis=1)

        # For each component, select series with earliest first_anomaly_timestamp; tie-breaker higher severity
        # Remove rows with NA first_anomaly_timestamp if any
        agg_anom = agg_anom[pd.notna(agg_anom['first_anomaly_timestamp'])].copy()
        agg_anom['first_anomaly_timestamp'] = agg_anom['first_anomaly_timestamp'].astype(int)

        # Sort for grouping: earliest first_anomaly_timestamp asc, severity desc
        agg_anom_sorted = agg_anom.sort_values(['cmdb_id','first_anomaly_timestamp','severity'], ascending=[True, True, False])

        # Pick one per component
        selected_list = []
        for comp in candidates:
            comp_rows = agg_anom_sorted[agg_anom_sorted['cmdb_id']==comp]
            if comp_rows.empty:
                continue
            # pick rows with minimal first timestamp
            min_ts = comp_rows['first_anomaly_timestamp'].min()
            candidates_rows = comp_rows[comp_rows['first_anomaly_timestamp']==min_ts]
            # tie-breaker by severity desc
            chosen = candidates_rows.sort_values('severity', ascending=False).head(1)
            selected_list.append(chosen)

        if not selected_list:
            selected_rows = pd.DataFrame(columns=[
                'cmdb_id','kpi_name','anomaly_type','first_anomaly_timestamp','value_at_first_anomaly',
                'global_p95','global_p05','max_value_in_window','min_value_in_window','mean_value_in_window',
                'anomaly_count_in_window','longest_consecutive_anomaly_run'
            ])
            summary_line = "No anomalies detected for candidate components in the window."
            selected_rows, summary_line
        else:
            selected_df = pd.concat(selected_list, ignore_index=True)

            # Prepare final columns and limit to up to 5 rows, sorted by first_anomaly_timestamp ascending
            final_cols = [
                'cmdb_id','kpi_name','anomaly_type','first_anomaly_timestamp','value_at_first_anomaly',
                'global_p95','global_p05','max_value_in_window','min_value_in_window','mean_value_in_window',
                'anomaly_count_in_window','longest_consecutive_anomaly_run'
            ]
            selected_rows = selected_df[final_cols].sort_values('first_anomaly_timestamp').head(5).reset_index(drop=True)

            # Create summary line: earliest overall anomaly timestamp and which component produced it
            earliest_idx = selected_rows['first_anomaly_timestamp'].idxmin()
            earliest_ts = int(selected_rows.loc[earliest_idx,'first_anomaly_timestamp'])
            earliest_comp = selected_rows.loc[earliest_idx,'cmdb_id']
            summary_line = f"Earliest anomaly among selected components: timestamp {earliest_ts} (UTC) produced by component '{earliest_comp}'."

            # Return selected_rows and summary_line
            selected_rows, summary_line

```
Out[4]:
```
I can't summarize because the last execution returned None (no results were emitted). That means selected_rows and summary_line were not returned from the cell.

I can rerun the selection and explicitly return the two objects so you get the 5-row table and the summary line. Shall I rerun now?

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse previously loaded dataframes if available, otherwise load
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")

# Normalize log to kpi_name
df_log2 = df_log.copy()
if 'log_name' in df_log2.columns:
    df_log2 = df_log2.rename(columns={'log_name': 'kpi_name'})
else:
    df_log2['kpi_name'] = df_log2.get('kpi_name', None)

# Prepare combined dataframe (metric + log)
df_metric2 = df_metric.copy()
df_metric2['kpi_name'] = df_metric2['kpi_name'].astype(str)
df_log2['kpi_name'] = df_log2['kpi_name'].astype(str)

df_all = pd.concat([
    df_metric2[['timestamp','cmdb_id','kpi_name','value']].assign(__source='metric'),
    df_log2[['timestamp','cmdb_id','kpi_name','value']].assign(__source='log')
], ignore_index=True)

# Ensure numeric and parse timestamps as UTC datetimes (rule 10)
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')
df_all['timestamp'] = pd.to_numeric(df_all['timestamp'], errors='coerce').astype('Int64')
df_all['timestamp_dt'] = pd.to_datetime(df_all['timestamp'].astype('int64'), unit='s', utc=True)

# Candidate components
candidates = ["catalogue","carts","orders","payment","user"]

# Compute (or reuse cached) global P95 and P05 per (cmdb_id,kpi_name) using full series BEFORE filtering (rule 9)
if 'group_stats_full' in globals():
    group_stats = group_stats_full
else:
    group_stats = df_all.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p05=lambda x: x.quantile(0.05)
    ).reset_index()
    group_stats_full = group_stats  # cache for reuse

# Incident window (UTC)
window_start = pd.to_datetime('2024-01-18 18:10:12', utc=True)
window_end = pd.to_datetime('2024-01-18 18:40:12', utc=True)

# Filter to candidate components and merge thresholds
df_cand = df_all[df_all['cmdb_id'].isin(candidates)].copy()
df_cand = df_cand.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

# Filter points to the incident window
df_win = df_cand[(df_cand['timestamp_dt'] >= window_start) & (df_cand['timestamp_dt'] <= window_end)].copy()

# Prepare output structure if no data
final_cols = [
    'cmdb_id','kpi_name','anomaly_type','first_anomaly_timestamp','value_at_first_anomaly',
    'global_p95','global_p05','max_value_in_window','min_value_in_window','mean_value_in_window',
    'anomaly_count_in_window','longest_consecutive_anomaly_run'
]

if df_win.empty:
    selected_rows = pd.DataFrame(columns=final_cols)
    summary_line = "No data in incident window for candidate components."
    selected_rows, summary_line
else:
    # Determine low-anomaly KPI tokens
    low_kpi_tokens = ['workload','row_count','throughput','request','requests','rps']
    df_win['kpi_lower'] = df_win['kpi_name'].str.lower()

    # Sample-level anomaly flags
    df_win['high_flag'] = df_win['value'] >= df_win['global_p95']
    df_win['low_flag'] = df_win.apply(
        lambda r: (any(tok in r['kpi_lower'] for tok in low_kpi_tokens)
                   and pd.notna(r['value']) and pd.notna(r['global_p05'])
                   and r['value'] <= r['global_p05']),
        axis=1
    )
    df_win['anom_flag'] = df_win['high_flag'] | df_win['low_flag']

    # Aggregate basic stats per series within window
    grouped = df_win.groupby(['cmdb_id','kpi_name'])
    agg = grouped.agg(
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        mean_value_in_window=('value','mean'),
        anomaly_count_in_window=('anom_flag','sum'),
        window_count=('value','count')
    ).reset_index()

    # Compute first anomaly timestamp, value at first anomaly, anomaly_type per series
    def first_anomaly_info(row):
        subset = df_win[
            (df_win['cmdb_id']==row['cmdb_id']) &
            (df_win['kpi_name']==row['kpi_name']) &
            (df_win['anom_flag'])
        ]
        if subset.empty:
            return pd.Series({
                'first_anomaly_timestamp': pd.NA,
                'value_at_first_anomaly': np.nan,
                'anomaly_type': None
            })
        first = subset.sort_values('timestamp').iloc[0]
        has_high = subset['high_flag'].any()
        has_low = subset['low_flag'].any()
        if has_high and has_low:
            atype = 'both'
        elif has_high:
            atype = 'high'
        else:
            atype = 'low'
        return pd.Series({
            'first_anomaly_timestamp': int(first['timestamp']),
            'value_at_first_anomaly': float(first['value']),
            'anomaly_type': atype
        })

    first_info = agg.apply(first_anomaly_info, axis=1)
    agg = pd.concat([agg, first_info], axis=1)

    # Merge global thresholds
    agg = agg.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

    # Compute longest consecutive anomaly run per series (consecutive minutes where anom_flag True)
    def longest_run(row):
        subset = df_win[(df_win['cmdb_id']==row['cmdb_id']) & (df_win['kpi_name']==row['kpi_name'])].sort_values('timestamp')
        anom = subset[subset['anom_flag']].copy()
        if anom.empty:
            return 0
        anom['diff'] = anom['timestamp'].diff().fillna(9999).astype(int)
        anom['run_id'] = (anom['diff'] != 60).cumsum()
        run_lengths = anom.groupby('run_id').size()
        return int(run_lengths.max())

    agg['longest_consecutive_anomaly_run'] = agg.apply(longest_run, axis=1).astype(int)

    # Keep only series that have at least one anomaly in window
    agg_anom = agg[agg['anomaly_count_in_window'] > 0].copy()
    if agg_anom.empty:
        selected_rows = pd.DataFrame(columns=final_cols)
        summary_line = "No anomalies detected for candidate components in the window."
        selected_rows, summary_line
    else:
        # Compute severity for tie-breaking
        def compute_severity(row):
            sev_high = np.nan
            sev_low = np.nan
            if row['anomaly_type'] in ('high','both'):
                if pd.notna(row['global_p95']) and row['global_p95'] != 0:
                    sev_high = row['max_value_in_window'] / row['global_p95']
            if row['anomaly_type'] in ('low','both'):
                if pd.notna(row['global_p05']):
                    if row['min_value_in_window'] == 0:
                        sev_low = np.inf
                    else:
                        sev_low = row['global_p05'] / row['min_value_in_window']
            sevs = [s for s in (sev_high, sev_low) if pd.notna(s)]
            return float(np.nanmax(sevs)) if sevs else np.nan

        agg_anom['severity'] = agg_anom.apply(compute_severity, axis=1)

        # Remove rows missing first_anomaly_timestamp
        agg_anom = agg_anom[pd.notna(agg_anom['first_anomaly_timestamp'])].copy()
        agg_anom['first_anomaly_timestamp'] = agg_anom['first_anomaly_timestamp'].astype(int)

        # For each component pick series with earliest first_anomaly_timestamp; tie-breaker higher severity
        selected_list = []
        for comp in candidates:
            comp_rows = agg_anom[agg_anom['cmdb_id'] == comp]
            if comp_rows.empty:
                continue
            min_ts = comp_rows['first_anomaly_timestamp'].min()
            tie_rows = comp_rows[comp_rows['first_anomaly_timestamp'] == min_ts]
            chosen = tie_rows.sort_values('severity', ascending=False).head(1)
            selected_list.append(chosen)

        if not selected_list:
            selected_rows = pd.DataFrame(columns=final_cols)
            summary_line = "No anomalies detected for candidate components in the window."
            selected_rows, summary_line
        else:
            selected_df = pd.concat(selected_list, ignore_index=True)
            # Prepare final compact table with requested columns
            selected_rows = selected_df[final_cols].sort_values('first_anomaly_timestamp').head(5).reset_index(drop=True)
            # summary line: earliest overall anomaly among selected rows
            earliest_idx = selected_rows['first_anomaly_timestamp'].idxmin()
            earliest_ts = int(selected_rows.loc[earliest_idx, 'first_anomaly_timestamp'])
            earliest_comp = selected_rows.loc[earliest_idx, 'cmdb_id']
            summary_line = f"Earliest anomaly among selected components: timestamp {earliest_ts} (UTC) produced by component '{earliest_comp}'."

            # Explicitly return the two objects as the final expression
            selected_rows, summary_line

```
Out[5]:
```
import pandas as pd
import numpy as np

# Load data (reuse if already in kernel)
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")

# Normalize log to kpi_name
df_log2 = df_log.copy()
if 'log_name' in df_log2.columns:
    df_log2 = df_log2.rename(columns={'log_name': 'kpi_name'})
else:
    df_log2['kpi_name'] = df_log2.get('kpi_name', None)

# Combine metric and log
df_metric2 = df_metric.copy()
df_metric2['kpi_name'] = df_metric2['kpi_name'].astype(str)
df_log2['kpi_name'] = df_log2['kpi_name'].astype(str)
df_all = pd.concat([
    df_metric2[['timestamp','cmdb_id','kpi_name','value']].assign(__source='metric'),
    df_log2[['timestamp','cmdb_id','kpi_name','value']].assign(__source='log')
], ignore_index=True)

# Types and timestamps
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')
df_all['timestamp'] = pd.to_numeric(df_all['timestamp'], errors='coerce').astype('Int64')
df_all['timestamp_dt'] = pd.to_datetime(df_all['timestamp'].astype('int64'), unit='s', utc=True)

# Candidates and window
candidates = ["catalogue","carts","orders","payment","user"]
window_start = pd.to_datetime('2024-01-18 18:10:12', utc=True)
window_end = pd.to_datetime('2024-01-18 18:40:12', utc=True)

# Compute global P95 and P05 per series (full series)
group_stats = df_all.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_p95=lambda x: x.quantile(0.95),
    global_p05=lambda x: x.quantile(0.05)
).reset_index()

# Filter to candidates and window, merge thresholds
df_cand = df_all[df_all['cmdb_id'].isin(candidates)].copy()
df_cand = df_cand.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')
df_win = df_cand[(df_cand['timestamp_dt'] >= window_start) & (df_cand['timestamp_dt'] <= window_end)].copy()

summary = ""
if df_win.empty:
    summary = "No data in the incident window for the candidate components."
else:
    # Define low-anomaly tokens
    low_kpi_tokens = ['workload','row_count','throughput','request','requests','rps']
    df_win['kpi_lower'] = df_win['kpi_name'].str.lower()

    # Flags
    df_win['high_flag'] = df_win['value'] >= df_win['global_p95']
    df_win['low_flag'] = df_win.apply(
        lambda r: (any(tok in r['kpi_lower'] for tok in low_kpi_tokens)
                   and pd.notna(r['value']) and pd.notna(r['global_p05'])
                   and r['value'] <= r['global_p05']),
        axis=1
    )
    df_win['anom_flag'] = df_win['high_flag'] | df_win['low_flag']

    # Aggregate per series
    grouped = df_win.groupby(['cmdb_id','kpi_name'])
    agg = grouped.agg(
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        mean_value_in_window=('value','mean'),
        anomaly_count_in_window=('anom_flag','sum'),
        window_count=('value','count')
    ).reset_index()

    # First anomaly info
    def first_info(row):
        subset = df_win[(df_win['cmdb_id']==row['cmdb_id']) & (df_win['kpi_name']==row['kpi_name']) & (df_win['anom_flag'])]
        if subset.empty:
            return pd.Series({'first_anomaly_timestamp': pd.NA, 'value_at_first_anomaly': np.nan, 'anomaly_type': None})
        first = subset.sort_values('timestamp').iloc[0]
        has_high = subset['high_flag'].any()
        has_low = subset['low_flag'].any()
        if has_high and has_low:
            atype = 'both'
        elif has_high:
            atype = 'high'
        else:
            atype = 'low'
        return pd.Series({'first_anomaly_timestamp': int(first['timestamp']), 'value_at_first_anomaly': float(first['value']), 'anomaly_type': atype})

    first_df = agg.apply(first_info, axis=1)
    agg = pd.concat([agg, first_df], axis=1)
    agg = agg.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

    # Longest consecutive anomaly run
    def longest_run(row):
        subset = df_win[(df_win['cmdb_id']==row['cmdb_id']) & (df_win['kpi_name']==row['kpi_name'])].sort_values('timestamp')
        anom = subset[subset['anom_flag']].copy()
        if anom.empty:
            return 0
        anom['diff'] = anom['timestamp'].diff().fillna(9999).astype(int)
        anom['run_id'] = (anom['diff'] != 60).cumsum()
        return int(anom.groupby('run_id').size().max())

    agg['longest_consecutive_anomaly_run'] = agg.apply(longest_run, axis=1).astype(int)

    # Keep series with >=1 anomaly
    agg_anom = agg[agg['anomaly_count_in_window'] > 0].copy()
    if agg_anom.empty:
        summary = "No anomalies detected for candidate components in the window."
    else:
        # Severity for tie-breaker
        def severity(row):
            sev_high = np.nan
            sev_low = np.nan
            if row['anomaly_type'] in ('high','both'):
                if pd.notna(row['global_p95']) and row['global_p95'] != 0:
                    sev_high = row['max_value_in_window'] / row['global_p95']
            if row['anomaly_type'] in ('low','both'):
                if pd.notna(row['global_p05']):
                    if row['min_value_in_window'] == 0:
                        sev_low = np.inf
                    else:
                        sev_low = row['global_p05'] / row['min_value_in_window']
            sevs = [s for s in (sev_high, sev_low) if pd.notna(s)]
            return float(np.nanmax(sevs)) if sevs else np.nan

        agg_anom['severity'] = agg_anom.apply(severity, axis=1)
        agg_anom = agg_anom[pd.notna(agg_anom['first_anomaly_timestamp'])].copy()
        agg_anom['first_anomaly_timestamp'] = agg_anom['first_anomaly_timestamp'].astype(int)

        # Select single series per component: earliest first_anomaly_timestamp, tie-break by severity
        selected = []
        for comp in candidates:
            comp_rows = agg_anom[agg_anom['cmdb_id']==comp]
            if comp_rows.empty:
                continue
            min_ts = comp_rows['first_anomaly_timestamp'].min()
            tie = comp_rows[comp_rows['first_anomaly_timestamp']==min_ts]
            chosen = tie.sort_values('severity', ascending=False).head(1)
            selected.append(chosen)
        if not selected:
            summary = "No anomalies detected for candidate components in the window."
        else:
            sel_df = pd.concat(selected, ignore_index=True)
            final_cols = [
                'cmdb_id','kpi_name','anomaly_type','first_anomaly_timestamp','value_at_first_anomaly',
                'global_p95','global_p05','max_value_in_window','min_value_in_window','mean_value_in_window',
                'anomaly_count_in_window','longest_consecutive_anomaly_run'
            ]
            selected_rows = sel_df[final_cols].sort_values('first_anomaly_timestamp').reset_index(drop=True)
            # Compose compact CSV-like table as string
            csv_lines = [",".join(final_cols)]
            for _, r in selected_rows.iterrows():
                row_vals = [
                    str(r['cmdb_id']),
                    str(r['kpi_name']),
                    str(r['anomaly_type']),
                    str(int(r['first_anomaly_timestamp'])),
                    f"{r['value_at_first_anomaly']:.6g}",
                    f"{r['global_p95']:.6g}",
                    f"{r['global_p05']:.6g}",
                    f"{r['max_value_in_window']:.6g}",
                    f"{r['min_value_in_window']:.6g}",
                    f"{r['mean_value_in_window']:.6g}",
                    str(int(r['anomaly_count_in_window'])),
                    str(int(r['longest_consecutive_anomaly_run']))
                ]
                csv_lines.append(",".join(row_vals))
            table_str = "\n".join(csv_lines)
            # summary line
            earliest_idx = selected_rows['first_anomaly_timestamp'].idxmin()
            earliest_ts = int(selected_rows.loc[earliest_idx, 'first_anomaly_timestamp'])
            earliest_comp = selected_rows.loc[earliest_idx, 'cmdb_id']
            summary = f"Selected series (one per component):\n{table_str}\n\nEarliest anomaly among these: timestamp {earliest_ts} (UTC) by component '{earliest_comp}'."

# Output the single summary string (plain English + compact CSV-like table if present)
summary

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse loaded dataframes if present, otherwise load
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")

# Normalize log to kpi_name
df_log2 = df_log.copy()
if 'log_name' in df_log2.columns:
    df_log2 = df_log2.rename(columns={'log_name': 'kpi_name'})
else:
    df_log2['kpi_name'] = df_log2.get('kpi_name', None)

# Combine metric and log into unified df_all
df_metric2 = df_metric.copy()
df_metric2['kpi_name'] = df_metric2['kpi_name'].astype(str)
df_log2['kpi_name'] = df_log2['kpi_name'].astype(str)

df_all = pd.concat([
    df_metric2[['timestamp','cmdb_id','kpi_name','value']].assign(__source='metric'),
    df_log2[['timestamp','cmdb_id','kpi_name','value']].assign(__source='log')
], ignore_index=True)

# Ensure numeric types and parse timestamps to UTC datetimes
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')
df_all['timestamp'] = pd.to_numeric(df_all['timestamp'], errors='coerce').astype('Int64')
df_all['timestamp_dt'] = pd.to_datetime(df_all['timestamp'].astype('int64'), unit='s', utc=True)

# Candidate components and incident window
candidates = ["catalogue","carts","orders","payment","user"]
window_start = pd.to_datetime('2024-01-18 18:10:12', utc=True)
window_end = pd.to_datetime('2024-01-18 18:40:12', utc=True)

# Compute global P95 and P05 per (cmdb_id,kpi_name) using full series (cache if previously computed)
if 'group_stats_full' in globals():
    group_stats = group_stats_full
else:
    group_stats = df_all.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p05=lambda x: x.quantile(0.05)
    ).reset_index()
    group_stats_full = group_stats  # cache for reuse

# Filter to candidate components and merge thresholds
df_cand = df_all[df_all['cmdb_id'].isin(candidates)].copy()
df_cand = df_cand.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

# Filter to incident window
df_win = df_cand[(df_cand['timestamp_dt'] >= window_start) & (df_cand['timestamp_dt'] <= window_end)].copy()

# Prepare final columns
final_cols = [
    'cmdb_id','kpi_name','anomaly_type','first_anomaly_timestamp','value_at_first_anomaly',
    'global_p95','global_p05','max_value_in_window','min_value_in_window','mean_value_in_window',
    'anomaly_count_in_window','longest_consecutive_anomaly_run'
]

# If no data in window, return empty DataFrame and summary
if df_win.empty:
    selected_rows = pd.DataFrame(columns=final_cols)
    summary_line = "No data in the incident window for the candidate components."
    selected_rows, summary_line
else:
    # Low-anomaly tokens
    low_kpi_tokens = ['workload','row_count','throughput','request','requests','rps']
    df_win['kpi_lower'] = df_win['kpi_name'].str.lower()

    # Sample-level anomaly flags
    df_win['high_flag'] = df_win['value'] >= df_win['global_p95']
    df_win['low_flag'] = df_win.apply(
        lambda r: (any(tok in r['kpi_lower'] for tok in low_kpi_tokens)
                   and pd.notna(r['value']) and pd.notna(r['global_p05'])
                   and r['value'] <= r['global_p05']),
        axis=1
    )
    df_win['anom_flag'] = df_win['high_flag'] | df_win['low_flag']

    # Aggregate per series within window
    grouped = df_win.groupby(['cmdb_id','kpi_name'])
    agg = grouped.agg(
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        mean_value_in_window=('value','mean'),
        anomaly_count_in_window=('anom_flag','sum'),
        window_count=('value','count')
    ).reset_index()

    # Compute first anomaly info and anomaly_type for each series
    def first_anomaly_info(row):
        subset = df_win[
            (df_win['cmdb_id']==row['cmdb_id']) &
            (df_win['kpi_name']==row['kpi_name']) &
            (df_win['anom_flag'])
        ]
        if subset.empty:
            return pd.Series({'first_anomaly_timestamp': pd.NA, 'value_at_first_anomaly': np.nan, 'anomaly_type': None})
        first = subset.sort_values('timestamp').iloc[0]
        has_high = subset['high_flag'].any()
        has_low = subset['low_flag'].any()
        if has_high and has_low:
            atype = 'both'
        elif has_high:
            atype = 'high'
        else:
            atype = 'low'
        return pd.Series({
            'first_anomaly_timestamp': int(first['timestamp']),
            'value_at_first_anomaly': float(first['value']),
            'anomaly_type': atype
        })

    first_info = agg.apply(first_anomaly_info, axis=1)
    agg = pd.concat([agg, first_info], axis=1)

    # Merge global thresholds
    agg = agg.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

    # Longest consecutive anomaly run per series
    def longest_run(row):
        subset = df_win[(df_win['cmdb_id']==row['cmdb_id']) & (df_win['kpi_name']==row['kpi_name'])].sort_values('timestamp')
        anom = subset[subset['anom_flag']].copy()
        if anom.empty:
            return 0
        anom['diff'] = anom['timestamp'].diff().fillna(9999).astype(int)
        anom['run_id'] = (anom['diff'] != 60).cumsum()
        return int(anom.groupby('run_id').size().max())

    agg['longest_consecutive_anomaly_run'] = agg.apply(longest_run, axis=1).astype(int)

    # Keep only series with >=1 anomaly
    agg_anom = agg[agg['anomaly_count_in_window'] > 0].copy()
    if agg_anom.empty:
        selected_rows = pd.DataFrame(columns=final_cols)
        summary_line = "No anomalies detected for candidate components in the window."
        selected_rows, summary_line
    else:
        # Compute severity for tie-breaker
        def compute_severity(row):
            sev_high = np.nan
            sev_low = np.nan
            if row['anomaly_type'] in ('high','both'):
                if pd.notna(row['global_p95']) and row['global_p95'] != 0:
                    sev_high = row['max_value_in_window'] / row['global_p95']
            if row['anomaly_type'] in ('low','both'):
                if pd.notna(row['global_p05']):
                    if row['min_value_in_window'] == 0:
                        sev_low = np.inf
                    else:
                        sev_low = row['global_p05'] / row['min_value_in_window']
            sevs = [s for s in (sev_high, sev_low) if pd.notna(s)]
            return float(np.nanmax(sevs)) if sevs else np.nan

        agg_anom['severity'] = agg_anom.apply(compute_severity, axis=1)

        # Drop entries without a first anomaly timestamp
        agg_anom = agg_anom[pd.notna(agg_anom['first_anomaly_timestamp'])].copy()
        agg_anom['first_anomaly_timestamp'] = agg_anom['first_anomaly_timestamp'].astype(int)

        # For each component select the series with earliest first_anomaly_timestamp; tie-breaker higher severity
        selected_list = []
        for comp in candidates:
            comp_rows = agg_anom[agg_anom['cmdb_id'] == comp]
            if comp_rows.empty:
                continue
            min_ts = comp_rows['first_anomaly_timestamp'].min()
            tie_rows = comp_rows[comp_rows['first_anomaly_timestamp'] == min_ts]
            chosen = tie_rows.sort_values('severity', ascending=False).head(1)
            selected_list.append(chosen)

        # Build selected_rows DataFrame (one per component), sorted by first_anomaly_timestamp ascending
        if not selected_list:
            selected_rows = pd.DataFrame(columns=final_cols)
            summary_line = "No anomalies detected for candidate components in the window."
        else:
            selected_df = pd.concat(selected_list, ignore_index=True)
            selected_rows = selected_df[final_cols].sort_values('first_anomaly_timestamp').reset_index(drop=True).head(5)

            # Compose summary_line
            earliest_idx = selected_rows['first_anomaly_timestamp'].idxmin()
            earliest_ts = int(selected_rows.loc[earliest_idx, 'first_anomaly_timestamp'])
            earliest_comp = selected_rows.loc[earliest_idx, 'cmdb_id']
            summary_line = f"Earliest anomaly among selected components: timestamp {earliest_ts} (UTC) produced by component '{earliest_comp}'."

# Explicitly return the two required objects for the IPython kernel
selected_rows, summary_line

```
Out[6]:
```
Summary (one selected series per component, earliest anomaly in-window shown):

- catalogue — kpi: log.error_count — anomaly_type: high  
  first_anomaly_timestamp: 1705601580 (UTC)  
  value_at_first_anomaly: 0.0  
  global_p95: 0.0, global_p05: 0.0  
  max/min/mean in window: 0.0 / 0.0 / 0.0  
  anomaly_count_in_window: 21, longest_consecutive_anomaly_run: 14

- carts — kpi: log.row_count — anomaly_type: both  
  first_anomaly_timestamp: 1705601580 (UTC)  
  value_at_first_anomaly: 25.0  
  global_p95: 131.25, global_p05: 25.0  
  max/min/mean in window: 150.0 / 25.0 / 68.75  
  anomaly_count_in_window: 5, longest_consecutive_anomaly_run: 2

- orders — kpi: error — anomaly_type: high  
  first_anomaly_timestamp: 1705601580 (UTC)  
  value_at_first_anomaly: 0.0  
  global_p95: 0.0, global_p05: 0.0  
  max/min/mean in window: 0.0 / 0.0 / 0.0  
  anomaly_count_in_window: 25, longest_consecutive_anomaly_run: 25

- payment — kpi: socket — anomaly_type: high  
  first_anomaly_timestamp: 1705601580 (UTC)  
  value_at_first_anomaly: 5.0  
  global_p95: 5.0, global_p05: 5.0  
  max/min/mean in window: 5.0 / 5.0 / 5.0  
  anomaly_count_in_window: 25, longest_consecutive_anomaly_run: 25

- user — kpi: latency-90 — anomaly_type: high  
  first_anomaly_timestamp: 1705601580 (UTC)  
  value_at_first_anomaly: 0.005418  
  global_p95: 0.005406, global_p05: 0.004762  
  max/min/mean in window: 0.005702 / 0.00475 / 0.005074  
  anomaly_count_in_window: 2, longest_consecutive_anomaly_run: 1

Earliest anomaly among these rows: timestamp 1705601580 (UTC), produced by component 'catalogue'.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id         kpi_name anomaly_type  first_anomaly_timestamp  value_at_first_anomaly  global_p95  global_p05  max_value_in_window  min_value_in_window  mean_value_in_window  anomaly_count_in_window  longest_consecutive_anomaly_run
0  catalogue  log.error_count         high               1705601580                0.000000    0.000000    0.000000             0.000000              0.00000              0.000000                       21                               14
1      carts    log.row_count         both               1705601580               25.000000  131.250000   25.000000           150.000000             25.00000             68.750000                        5                                2
2     orders            error         high               1705601580                0.000000    0.000000    0.000000             0.000000              0.00000              0.000000                       25                               25
3    payment           socket         high               1705601580                5.000000    5.000000    5.000000             5.000000              5.00000              5.000000                       25                               25
4       user       latency-90         high               1705601580                0.005418    0.005406    0.004762             0.005702              0.00475              0.005074                        2                                1, "Earliest anomaly among selected components: timestamp 1705601580 (UTC) produced by component 'catalogue'.")```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Load or reuse dataframes
if 'df_metric' not in globals():
    df_metric = pd.read_csv("metric.csv")
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")

# Normalize log to kpi_name
df_log2 = df_log.copy()
if 'log_name' in df_log2.columns:
    df_log2 = df_log2.rename(columns={'log_name': 'kpi_name'})
else:
    df_log2['kpi_name'] = df_log2.get('kpi_name', None)

# Combine metric and log
df_metric2 = df_metric.copy()
df_metric2['kpi_name'] = df_metric2['kpi_name'].astype(str)
df_log2['kpi_name'] = df_log2['kpi_name'].astype(str)

df_all = pd.concat([
    df_metric2[['timestamp','cmdb_id','kpi_name','value']].assign(__source='metric'),
    df_log2[['timestamp','cmdb_id','kpi_name','value']].assign(__source='log')
], ignore_index=True)

# Ensure numeric and parse timestamps (UTC)
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')
df_all['timestamp'] = pd.to_numeric(df_all['timestamp'], errors='coerce').astype('Int64')
df_all['timestamp_dt'] = pd.to_datetime(df_all['timestamp'].astype('int64'), unit='s', utc=True)

# Candidate components and incident window
candidates = ["catalogue","carts","orders","payment","user"]
window_start = pd.to_datetime('2024-01-18 18:10:12', utc=True)
window_end = pd.to_datetime('2024-01-18 18:40:12', utc=True)

# Compute global P95 and P05 per series using full series (before filtering)
group_stats = df_all.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_p95=lambda x: x.quantile(0.95),
    global_p05=lambda x: x.quantile(0.05)
).reset_index()

# Filter to candidate components and merge thresholds
df_cand = df_all[df_all['cmdb_id'].isin(candidates)].copy()
df_cand = df_cand.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

# Filter to incident window
df_win = df_cand[(df_cand['timestamp_dt'] >= window_start) & (df_cand['timestamp_dt'] <= window_end)].copy()

# If no data in window, return empty outputs
final_cols = [
    'cmdb_id','kpi_name','anomaly_type','first_anomaly_timestamp','value_at_first_anomaly',
    'global_p95','global_p05','max_value_in_window','min_value_in_window','mean_value_in_window',
    'anomaly_count_in_window','longest_consecutive_anomaly_run','severity'
]

if df_win.empty:
    table_str = ",".join(final_cols)  # header only
    summary_line = "No data in the incident window for candidate components."
    table_str, summary_line
else:
    # Define low-anomaly KPI tokens
    low_kpi_tokens = ['workload','row_count','throughput','request','requests','rps']
    df_win['kpi_lower'] = df_win['kpi_name'].str.lower()

    # Sample-level anomaly flags
    df_win['high_flag'] = df_win['value'] >= df_win['global_p95']
    df_win['low_flag'] = df_win.apply(
        lambda r: (any(tok in r['kpi_lower'] for tok in low_kpi_tokens)
                   and pd.notna(r['value']) and pd.notna(r['global_p05'])
                   and r['value'] <= r['global_p05']),
        axis=1
    )
    df_win['anom_flag'] = df_win['high_flag'] | df_win['low_flag']

    # Aggregate per series within window
    grp = df_win.groupby(['cmdb_id','kpi_name'])
    agg = grp.agg(
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        mean_value_in_window=('value','mean'),
        anomaly_count_in_window=('anom_flag','sum'),
        window_count=('value','count')
    ).reset_index()

    # Compute first anomaly info and anomaly_type
    def first_anom_info(row):
        subset = df_win[
            (df_win['cmdb_id']==row['cmdb_id']) &
            (df_win['kpi_name']==row['kpi_name']) &
            (df_win['anom_flag'])
        ]
        if subset.empty:
            return pd.Series({'first_anomaly_timestamp': pd.NA, 'value_at_first_anomaly': np.nan, 'anomaly_type': None})
        first = subset.sort_values('timestamp').iloc[0]
        has_high = subset['high_flag'].any()
        has_low = subset['low_flag'].any()
        if has_high and has_low:
            atype = 'both'
        elif has_high:
            atype = 'high'
        else:
            atype = 'low'
        return pd.Series({
            'first_anomaly_timestamp': int(first['timestamp']),
            'value_at_first_anomaly': float(first['value']),
            'anomaly_type': atype
        })

    first_info = agg.apply(first_anom_info, axis=1)
    agg = pd.concat([agg, first_info], axis=1)

    # Merge thresholds
    agg = agg.merge(group_stats, on=['cmdb_id','kpi_name'], how='left')

    # Compute longest consecutive anomaly run
    def longest_run(row):
        subset = df_win[(df_win['cmdb_id']==row['cmdb_id']) & (df_win['kpi_name']==row['kpi_name'])].sort_values('timestamp')
        anom = subset[subset['anom_flag']].copy()
        if anom.empty:
            return 0
        anom['diff'] = anom['timestamp'].diff().fillna(9999).astype(int)
        anom['run_id'] = (anom['diff'] != 60).cumsum()
        return int(anom.groupby('run_id').size().max())

    agg['longest_consecutive_anomaly_run'] = agg.apply(longest_run, axis=1).astype(int)

    # Keep only series with at least one anomaly in window
    agg_anom = agg[agg['anomaly_count_in_window'] > 0].copy()
    if agg_anom.empty:
        table_str = ",".join(final_cols)
        summary_line = "No anomalies detected for candidate components in the window."
        table_str, summary_line
    else:
        # Compute severity_high and severity_low and series severity
        def compute_severities(row):
            sev_high = -np.inf
            sev_low = -np.inf
            # high severity if global_p95 > 0
            if pd.notna(row['global_p95']) and row['global_p95'] > 0:
                sev_high = row['max_value_in_window'] / row['global_p95']
            # low severity if global_p05 > 0 and min_value_in_window > 0
            if pd.notna(row['global_p05']) and row['global_p05'] > 0 and pd.notna(row['min_value_in_window']) and row['min_value_in_window'] > 0:
                sev_low = row['global_p05'] / row['min_value_in_window']
            # series severity is max of the two (treat -inf as missing)
            severity = max(sev_high, sev_low)
            # If severity is -inf or not finite, return NaN to exclude later
            if not np.isfinite(severity) or severity <= 0:
                return np.nan
            return float(severity)

        agg_anom['severity'] = agg_anom.apply(compute_severities, axis=1)

        # Exclude series with non-finite/NaN severity
        agg_anom_finite = agg_anom[pd.notna(agg_anom['severity']) & np.isfinite(agg_anom['severity'])].copy()
        if agg_anom_finite.empty:
            table_str = ",".join(final_cols)
            summary_line = "No series with finite positive severity for candidate components."
            table_str, summary_line
        else:
            # For each component select series with highest severity; tie-breaker: longest_consecutive_anomaly_run desc, then anomaly_count_in_window desc
            selected_rows_list = []
            for comp in candidates:
                comp_rows = agg_anom_finite[agg_anom_finite['cmdb_id'] == comp].copy()
                if comp_rows.empty:
                    continue
                comp_rows_sorted = comp_rows.sort_values(
                    by=['severity','longest_consecutive_anomaly_run','anomaly_count_in_window'],
                    ascending=[False, False, False]
                )
                selected_rows_list.append(comp_rows_sorted.head(1))
            if not selected_rows_list:
                table_str = ",".join(final_cols)
                summary_line = "No series selected for candidate components."
                table_str, summary_line
            else:
                selected_df = pd.concat(selected_rows_list, ignore_index=True)
                # Limit to up to 5 rows and sort by severity desc
                selected_df = selected_df.sort_values('severity', ascending=False).head(5).reset_index(drop=True)

                # Build CSV-like string
                csv_lines = [",".join(final_cols)]
                for _, r in selected_df.iterrows():
                    vals = [
                        str(r['cmdb_id']),
                        str(r['kpi_name']),
                        str(r['anomaly_type']),
                        str(int(r['first_anomaly_timestamp'])) if pd.notna(r['first_anomaly_timestamp']) else "",
                        f"{r['value_at_first_anomaly']:.6g}" if pd.notna(r['value_at_first_anomaly']) else "",
                        f"{r['global_p95']:.6g}" if pd.notna(r['global_p95']) else "",
                        f"{r['global_p05']:.6g}" if pd.notna(r['global_p05']) else "",
                        f"{r['max_value_in_window']:.6g}" if pd.notna(r['max_value_in_window']) else "",
                        f"{r['min_value_in_window']:.6g}" if pd.notna(r['min_value_in_window']) else "",
                        f"{r['mean_value_in_window']:.6g}" if pd.notna(r['mean_value_in_window']) else "",
                        str(int(r['anomaly_count_in_window'])),
                        str(int(r['longest_consecutive_anomaly_run'])),
                        f"{r['severity']:.6g}"
                    ]
                    csv_lines.append(",".join(vals))
                table_str = "\n".join(csv_lines)

                # Summary line: component with highest severity (first row)
                top = selected_df.sort_values('severity', ascending=False).iloc[0]
                summary_line = (f"Likely root-cause candidate: component '{top['cmdb_id']}' with kpi '{top['kpi_name']}', "
                                f"severity={top['severity']:.6g}.")

# Explicitly return the two objects so the kernel emits them
table_str, summary_line

```
Out[7]:
```
Summary of selected series (one per component), sorted by severity:

- catalogue — kpi: log.row_count — anomaly_type: both — first_anomaly_timestamp: 1705601880 (UTC) — value_at_first_anomaly: 167 — global_p95: 167 — global_p05: 28 — max: 168 — min: 2 — mean: 125.095 — anomaly_count: 5 — longest_run: 1 — severity: 14  
- user — kpi: log.row_count — anomaly_type: both — first_anomaly_timestamp: 1705601940 (UTC) — value_at_first_anomaly: 613 — global_p95: 612.4 — global_p05: 40 — max: 619 — min: 8 — mean: 373.88 — anomaly_count: 7 — longest_run: 3 — severity: 5  
- payment — kpi: log.row_count — anomaly_type: both — first_anomaly_timestamp: 1705601940 (UTC) — value_at_first_anomaly: 107 — global_p95: 104 — global_p05: 40 — max: 107 — min: 8 — mean: 75.2 — anomaly_count: 10 — longest_run: 5 — severity: 5  
- orders — kpi: log.row_count — anomaly_type: both — first_anomaly_timestamp: 1705601940 (UTC) — value_at_first_anomaly: 134 — global_p95: 128.9 — global_p05: 22.2 — max: 134 — min: 12 — mean: 102.222 — anomaly_count: 2 — longest_run: 1 — severity: 1.85  
- carts — kpi: latency-50 — anomaly_type: high — first_anomaly_timestamp: 1705602540 (UTC) — value_at_first_anomaly: 0.0201213 — global_p95: 0.0195273 — global_p05: 0.014233 — max: 0.0201213 — min: 0.00954365 — mean: 0.0182866 — anomaly_count: 2 — longest_run: 1 — severity: 1.49136

Summary line (likely root-cause candidate): component 'catalogue' with kpi 'log.row_count' is the top candidate (severity = 14).

The original code execution output of IPython Kernel is also provided below for reference:

('cmdb_id,kpi_name,anomaly_type,first_anomaly_timestamp,value_at_first_anomaly,global_p95,global_p05,max_value_in_window,min_value_in_window,mean_value_in_window,anomaly_count_in_window,longest_consecutive_anomaly_run,severity\ncatalogue,log.row_count,both,1705601880,167,167,28,168,2,125.095,5,1,14\nuser,log.row_count,both,1705601940,613,612.4,40,619,8,373.88,7,3,5\npayment,log.row_count,both,1705601940,107,104,40,107,8,75.2,10,5,5\norders,log.row_count,both,1705601940,134,128.9,22.2,134,12,102.222,2,1,1.85\ncarts,latency-50,high,1705602540,0.0201213,0.0195273,0.014233,0.0201213,0.00954365,0.0182866,2,1,1.49136', "Likely root-cause candidate: component 'catalogue' with kpi 'log.row_count', severity=14.")```
```